In [1]:
# 07 : Step 1: bootstrap + load fold 0 checkpoint
import sys, os, importlib
SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()

import numpy as np, pandas as pd, torch
import config as C, data as D, model as M

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_DIR = C.RESULTS / "runs" / "20260821_2337_baseline"
FOLD    = 0

D.setup_data()                                   # Grad-CAM opens real image files

net, meta = M.load_checkpoint(RUN_DIR / f"fold_{FOLD}" / "best.pt", device=DEVICE)
target_layer = net.features[-1]                  # last spatial layer, 7x7 grid

oof = pd.read_csv(RUN_DIR / f"fold_{FOLD}" / "oof_preds.csv")

print("checkpoint :", {k: meta[k] for k in ("fold", "epoch", "aug_mode", "img_size")})
print("held-out F1:", round(meta["metrics"]["macro_f1"], 4))
print("target layer:", type(target_layer).__name__,
      "->", target_layer[0].out_channels, "channels")
print("oof rows   :", len(oof), "| errors:", int((oof.y_pred != oof.y_true).sum()))
print("device     :", DEVICE, "| eval mode:", not net.training)

Mounted at /content/drive
restoring /content/drive/MyDrive/0_potato_project_v1/data/potato_raw/raw -> /content/potato
  Early_blight -> Potato___Early_blight
  Late_blight -> Potato___Late_blight
  Healthy -> Potato___healthy
checkpoint : {'fold': 0, 'epoch': 1, 'aug_mode': 'baseline', 'img_size': 224}
held-out F1: 0.9701
target layer: Conv2dNormActivation -> 960 channels
oof rows   : 429 | errors: 7
device     : cuda | eval mode: True


In [2]:
# 07: Step 2: Grad-CAM
import torch.nn.functional as F


class GradCAM:
    """Class-discriminative heatmaps from the last spatial layer.

    Weights each feature map by how strongly the class score responds to it,
    sums them, keeps positive evidence only, and upsamples to image size.
    """

    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.h = [
            layer.register_forward_hook(
                lambda m, i, o: setattr(self, "acts", o.detach())),
            layer.register_full_backward_hook(
                lambda m, gi, go: setattr(self, "grads", go[0].detach())),
        ]

    def __call__(self, x, class_idx=None):
        """x: (1,3,H,W) normalised tensor. Returns (cam HxW in [0,1], probs, idx)."""
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)                          # fp32: AMP off deliberately
        probs = torch.softmax(logits.float(), 1)[0].detach().cpu().numpy()
        idx = int(logits.argmax(1)) if class_idx is None else int(class_idx)

        logits[0, idx].backward()

        w = self.grads.mean(dim=(2, 3), keepdim=True)   # importance per channel
        cam = F.relu((w * self.acts).sum(1, keepdim=True))   # positive evidence only
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                            align_corners=False)[0, 0]

        cam = cam - cam.min()
        cam = cam / cam.max() if cam.max() > 0 else cam     # guard: all-zero map
        return cam.cpu().numpy(), probs, idx

    def close(self):
        for h in self.h:
            h.remove()


cam_engine = GradCAM(net, target_layer)
print("GradCAM attached to:", type(target_layer).__name__)
print("hooks registered   :", len(cam_engine.h))
print("note: model stays in eval mode; gradients flow but weights never update")

GradCAM attached to: Conv2dNormActivation
hooks registered   : 2
note: model stays in eval mode; gradients flow but weights never update


In [4]:
# 07 : Step 3: one image end-to-end
from PIL import Image

_, tf_eval = D.build_transforms(meta["aug_mode"])     # from checkpoint, not config
names = [C.IDX_TO_CLASS[i].replace("Potato___", "") for i in range(C.NUM_CLASSES)]


def load_tensor(rel_path):
    """Same preprocessing the model was evaluated under. Returns (1,3,224,224)."""
    with Image.open(C.DATA_ROOT / rel_path) as im:
        return tf_eval(im.convert("RGB")).unsqueeze(0).to(DEVICE)


row = oof[oof.y_pred == oof.y_true].iloc[0]
x = load_tensor(row.path)
cam, probs, idx = cam_engine(x)

print("image      :", row.path.split('/')[-1])
print("true       :", names[row.y_true], "| predicted:", names[idx])
print("probs      :", [round(float(p), 4) for p in probs])
print("csv probs  :", [round(row[f'p{i}'], 4) for i in range(3)])
print("match      :", np.allclose(probs, [row[f'p{i}'] for i in range(3)], atol=1e-3))
print()
print("cam shape  :", cam.shape, "| range:", f"[{cam.min():.3f}, {cam.max():.3f}]")
print("cam mean   :", f"{cam.mean():.3f}", "| non-zero:", f"{(cam > 0.01).mean():.1%}")

image      : 034959c1-f1e8-4a79-a6d5-3c1d14efa2f3___RS_Early.B 7136.JPG
true       : Early_blight | predicted: Early_blight
probs      : [1.0, 0.0, 0.0]
csv probs  : [np.float64(1.0), np.float64(0.0), np.float64(0.0)]
match      : True

cam shape  : (224, 224) | range: [0.000, 1.000]
cam mean   : 0.370 | non-zero: 91.6%


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# 07 — CELL 4: overlay + correct predictions, 3 per class
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.cm as cm_maps

MEAN = np.array(C.IMAGENET_MEAN).reshape(3, 1, 1)
STD  = np.array(C.IMAGENET_STD).reshape(3, 1, 1)


def to_image(x):
    """Undo ImageNet normalisation -> displayable HxWx3 array in [0,1]."""
    a = x[0].cpu().numpy() * STD + MEAN
    return np.clip(a, 0, 1).transpose(1, 2, 0)


def overlay(img, cam, alpha=0.45):
    heat = cm_maps.jet(cam)[..., :3]
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


correct = oof[oof.y_pred == oof.y_true]
picks = [correct[correct.y_true == c].head(3) for c in range(C.NUM_CLASSES)]

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for r, sub in enumerate(picks):
    for i, (_, row) in enumerate(sub.iterrows()):
        x = load_tensor(row.path)
        cam, probs, idx = cam_engine(x)
        img = to_image(x)

        axes[r, i*2].imshow(img); axes[r, i*2].axis("off")
        axes[r, i*2].set_title(f"{names[row.y_true]}", fontsize=9)
        axes[r, i*2+1].imshow(overlay(img, cam)); axes[r, i*2+1].axis("off")
        axes[r, i*2+1].set_title(f"p={probs[idx]:.3f}  hot={((cam>0.5).mean()):.0%}",
                                 fontsize=9)

plt.suptitle(f"Grad-CAM — correct predictions, fold {FOLD} held-out", fontsize=13)
plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [8]:
# 07 :Step 5: misclassified images, predicted vs true class maps
errs = oof[oof.y_pred != oof.y_true].reset_index(drop=True)
n = len(errs)

fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
axes = axes.reshape(n, 3)

for r, (_, row) in enumerate(errs.iterrows()):
    x = load_tensor(row.path)
    cam_p, probs, _ = cam_engine(x, class_idx=row.y_pred)    # evidence for the answer given
    cam_t, _, _     = cam_engine(x, class_idx=row.y_true)    # evidence for the right answer
    img = to_image(x)

    axes[r, 0].imshow(img)
    axes[r, 0].set_title(f"true {names[row.y_true]}", fontsize=9)
    axes[r, 1].imshow(overlay(img, cam_p))
    axes[r, 1].set_title(f"said {names[row.y_pred]}  p={probs[row.y_pred]:.2f}", fontsize=9)
    axes[r, 2].imshow(overlay(img, cam_t))
    axes[r, 2].set_title(f"for {names[row.y_true]}  p={probs[row.y_true]:.2f}", fontsize=9)
    for c in range(3):
        axes[r, c].axis("off")

plt.suptitle(f"Grad-CAM — {n} errors, fold {FOLD} held-out", fontsize=13)
plt.tight_layout(); plt.show()

print("\nERROR DETAIL")
for _, row in errs.iterrows():
    print(f"  {names[row.y_true]:>13} -> {names[row.y_pred]:<13} "
          f"conf {row.confidence:.3f}   {row.path.split('/')[-1][:40]}")
print(f"\nmean confidence on errors  : {errs.confidence.mean():.3f}")
print(f"mean confidence on correct : {oof[oof.y_pred == oof.y_true].confidence.mean():.3f}")

Output hidden; open in https://colab.research.google.com to view.